# ಪಾಠ 10 - ಉತ್ಪಾದನೆಯಲ್ಲಿ AI ಏಜೆಂಟ್ಗಳು

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework (Python) ಬಳಸಿಕೊಂಡು AI ಏಜೆಂಟ್ಗಳಿಗಾಗಿ **ಉತ್ಪಾದನಾ ಮಾದರಿಗಳನ್ನು** ಕಲಿಯುತ್ತೀರಿ. ನಾವು ಈ ಕೆಳಗಿನ ವಿಷಯಗಳನ್ನು ಒಳಗೊಂಡಿರುತ್ತೇವೆ:

- **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯ ಮತ್ತು ಲಾಗ್ ದಾಖಲೆ ಸೇರಿಸುವುದು
- **ಮೌಲ್ಯಮಾಪನ** — ಪ್ರತಿಕ್ರಿಯೆಗಳ ಗುಣಮಟ್ಟವನ್ನು ಅಂಕನ ಮಾಡಲು ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ಅನ್ನು ಬಳಸುವುದು
- **ಖರ್ಚು ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಆಪ್ಟಿಮೈಜೆಶನ್ ಮತ್ತು ಮಾದರಿ ಆಯ್ಕೆಗೆ ತಂತ್ರಗಳು

ಈ ದೃಶ್ಯವು ಬಳಕೆದಾರರಿಗೆ ಪ್ರವಾಸ ಯೋಜನೆ ಮಾಡಲು ಸಹಾಯ ಮಾಡುವ ಒಂದು **ಪ್ರಯಾಣ ಏಜೆಂಟ್** ಅನ್ನು ಸೂಚಿಸುತ್ತದೆ, ಮೇಲ್ಮಟ್ಟದಲ್ಲಿ ನಿಗಾ ಮತ್ತು ಮೌಲ್ಯಮಾಪನ ಸೇರಿಸಲಾಗಿದೆ.


## ಸಜ್ಜಿಕೆ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ಉತ್ಪಾದನಾ ಪರಿಗಣನೆಗಳು

AI ಏಜೆಂಟ್‌ಗಳನ್ನು ಪ್ರೋಟೋಟೈಪ್ಗಳಿಂದ ಉತ್ಪಾದನೆಗೆ ತರಲು ಮೂರು ಸ್ತಂಭಗಳಿಗೆ ಸೂಕ್ಷ್ಮ ಗಮನ ಅಗತ್ಯವಿದೆ:

1. **ನಿರೀಕ್ಷಣೀಯತೆ** — ನೀವು ಏಜೆಂಟ್ ಏನು ಮಾಡುತ್ತಿದೆ, ಅದಕ್ಕೆ ಎಷ್ಟು ಸಮಯ ಬೇಕಾಗುತ್ತದೆ ಮತ್ತು ಅದು ಯಾವ ಉಪಕರಣಗಳನ್ನು ಕರೆಸುತ್ತದೆಯೋ ಎಂಬದರ ಕುರಿತು ದೃಶ್ಯತೆ ಹೊಂದಿರಬೇಕು. ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಲಾಗಿಂಗ್ ಇಲ್ಲದೆ, ಉತ್ಪಾದನಾ ಸಮಸ್ಯೆಗಳ ಡಿಬಗಿಂಗ್ ಪ್ರಾಯೋಗಿಕವಾಗಿ ಅಸಾಧ್ಯವಾಗಿದೆ.

2. **ಮೌಲ್ಯಮಾಪನ** — ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟ ಪರೀಕ್ಷೆಗಳು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಕಾಲಕ್ರಮದಲ್ಲಿ ನಿಖರ, ಸಂಪೂರ್ಣ ಮತ್ತು ಸಹಾಯಕವಾಗಿಯೇ ಇರುವಂತೆ ಖಚಿತಪಡಿಸುತ್ತವೆ. ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ನಿರ್ದಿಷ್ಟ ಮಾನದಂಡಗಳ ವಿರುದ್ಧ ಪ್ರತಿಕ್ರಿಯೆಗಳಿಗೆ ಅಂಕಗಳನ್ನು ನೀಡಬಹುದು.

3. **ವೆಚ್ಚ ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಬಳಕೆಯು ನೇರವಾಗಿ ವೆಚ್ಚದ ಮೇಲೆ ಪ್ರಭಾವ ಹೊಂದುತ್ತದೆ. ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ, ಮಾದರಿ ಆಯ್ಕೆ ಮತ್ತು ಕ್ಯಾಶಿಂಗ್ಂತಹ ತಂತ್ರಗಳು ಗುಣಮಟ್ಟವನ್ನು ತ್ಯಾಗ ಮಾಡದೆ ವೆಚ್ಚವನ್ನು ನಿಯಂತ್ರಣದಲ್ಲಿಟ್ಟುಕೊಳ್ಳಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


## ನಿರೀಕ್ಷಣಾಸಾಧ್ಯ ಏಜೆಂಟ್ ನಿರ್ಮಿಸುವುದು

ನಾವು ಪ್ರಯಾಣಕ್ಕೆ ಸಂಬಂಧಿಸಿದ ಸಾಧನಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ ಮತ್ತು ವಿಲಂಬವನ್ನು ವೀಕ್ಷಿಸಲು ಏಜೆಂಟ್ ಕರೆಗಳನ್ನು ಸಮಯ ಅಳತೆಯೊಂದಿಗೆ ಕವರ್ ಮಾಡುತ್ತೇವೆ. ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು OpenTelemetry ಅಥವಾ ಅದೇ ರೀತಿಯ ಟ್ರೇಸಿಂಗ್ ಬ್ಯಾಕ್‌ಎಂಡ್ ಜೊತೆಗೆ ಏಕೀಕರಣ ಮಾಡುತ್ತೀರಿ.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ಮೌಲ್ಯಮಾಪನ ಮಾದರಿಗಳು

ಸಾಮಾನ್ಯ ಉತ್ಪಾದನಾ ಮಾದರಿಯೊಂದರಲ್ಲಿ ಎರಡನೇ ಏಜೆಂಟ್ ಅನ್ನು **ಮೌಲ್ಯಮಾಪಕ**ವಾಗಿ ಬಳಸಲಾಗುತ್ತದೆ. ಮೌಲ্যಮಾಪಕ ಪ್ರಾಥಮಿಕ ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆ ಮುಂತಾದ ಪೂರ್ವನಿರ್ಧಾರಿತ ಮಾನದಂಡಗಳ ವಿರುದ್ಧ ಅಂಕಮಾಪನ ಮಾಡುತ್ತಾನೆ.

ಇದರಿಂದ ಸಾಧ್ಯವಾಗುತ್ತದೆ:
- ಉತ್ತರಗಳು ಬಳಕೆದಾರರಿಗೆ ತಲುಪುವ ಮೊದಲು ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ಗೇಟುಗಳು
- ಪ್ರಾಂಪ್ಟ್‌ಗಳು ಅಥವಾ ಮಾದರಿಗಳು ಬದಲಾದಾಗ ರೆಗ್ರೆಷನ್ ಪತ್ತೆ
- ಕಾಲಕಾಲಕ್ಕೆ ಏಜೆಂಟ್‌ನ ಕಾರ್ಯಕ್ಷಮತೆಯ ನಿರಂತರ ಮೇಲ್ವಿಚಾರಣೆ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ಖರ್ಚು ನಿರ್ವಹಣೆ ತಂತ್ರಗಳು

ಉತ್ಪಾದನಾ ಎಐ ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ ವೆಚ್ಚವನ್ನು ನಿಯಂತ್ರಿಸುವುದು ಅತ್ಯಂತ ಮಹತ್ವವಾಗಿರುತ್ತದೆ. ಇಲ್ಲಿವೆ ಪ್ರಮುಖ ತಂತ್ರಗಳು:

| ತಂತ್ರ | ವಿವರಣೆ |
|---|---|
| **ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ** | ಸಿಸ್ಟಮ್ ಸೂಚನೆಗಳನ್ನು ಸಂಕ್ಷಿಪ್ತವಾಗಿರಿಸಿ. ಇನ್‌ಪುಟ್ ಟೋಕನ್‌ಗಳ ಸಂಖ್ಯೆಯನ್ನು ಕಡಿಮೆ ಮಾಡಲು ಅನಾವಶ್ಯಕ ಹಿನ್ನೆಲೆಯನ್ನು ತೆಗೆದುಹಾಕಿ. |
| **ಮಾದರಿ ಆಯ್ಕೆ** | ವರ್ಗೀಕರಣ ಅಥವಾ ಮಾಹಿತಿಯನ್ನು ಹೊರತಾಗಿಸುವಂತಹ ಸರಳ ಕೆಲಸಗಳಿಗೆ ಕಡಿಮೆ ಮತ್ತು ಸಸ್ತೆ ಮಾದರಿಗಳನ್ನು (ಉದಾ. GPT-4o-mini) ಬಳಸಿ, ಮತ್ತು ಸಂಕೀರ್ಣ ತರ್ಕಕ್ಕೆ ದೊಡ್ಡ ಮಾದರಿಗಳನ್ನು ಮೀಸಲಿಡಿ. |
| **ಕ್ಯಾಶಿಂಗ್** | ಉಪಕರಣದ ಫಲಿತಾಂಶಗಳು ಮತ್ತು ಅಲ್ವಾ ಪ್ರಶ್ನೆಗಳನ್ನು ಕ್ಯಾಶ್ ಮಾಡಿ ಪುನರಾವರ್ತಿತ API ಕರೆಗಳನ್ನು ತಪ್ಪಿಸಿ. |
| **ಟೋಕನ್ ಬಜೆಟ್‌ಗಳು** | ಅಪ್ರತೀಕ್ಷಿತವಾಗಿ ಉದ್ದವಾದ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ತಡೆಯಲು `max_tokens` ಮಿತಿಗಳನ್ನು ಹೊಂದಿಸಿ. |
| **ಬ್ಯಾಚಿಂಗ್** | ಸಾಧ್ಯವಾದಲ್ಲಿ ಅನೇಕ ಬಳಕೆದಾರ ಪ್ರಶ್ನೆಗಳನ್ನು ಒಂದೇ API ಕರೆಗೆ ಗುಂಪುಮಾಡಿ. |

ಪ್ರಾಯೋಗಿಕವಾಗಿ, ಹಂತಗೊಳಿಸಿದ ವಿಧಾನವು ಚೆನ್ನಾಗಿ ಕೆಲಸ ಮಾಡುತ್ತದೆ: ಸರಳ ವಿನಂತಿಗಳನ್ನು ವೇಗವಾಗಿ ಮತ್ತು ಕಡಿಮೆ ವೆಚ್ಚದ ಮಾದರಿಗೆ ನಿರ್ದೇಶಿಸಿ ಮತ್ತು ಕೇವಲ ಸಂಕೀರ್ಣ ವಿನಂತಿಗಳನ್ನು ಹೆಚ್ಚು ಸಾಮರ್ಥ್ಯದ ಮಾದರಿಗೆ ಮೇಲಕ್ಕೆ ಕಳುಹಿಸಿ.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕೆಳಕಂಡವುಗಳನ್ನು ಹೇಗೆ ಮಾಡುವುದನ್ನು ಕಲಿತಿರಿ:

1. **ಅವಲೋಕನ ಸಾಮರ್ಥ್ಯ ಸೇರಿಸಿ** ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯಮಾಪನ ಮತ್ತು ಲಾಗಿಂಗ್ ಮೂಲಕ, ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಮಾನಿಟರಿಂಗ್‌ಗಾಗಿ ನೆಲ ತಯಾರಿಸುವಂತೆ.
2. **ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಮೌಲ್ಯಮಾಪನ ಮಾಡಿ** ಸಂಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕಾರಿತ್ವಕ್ಕೆ ಅಂಕ ನೀಡುವ ಮೌಲ್ಯನಿರ್ಣಾಯಕ ಏಜೆಂಟ್ ಬಳಸಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿ.
3. **ವೆಚ್ಚವನ್ನು ನಿರ್ವಹಿಸಿ** ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ, ಮಾದರಿ ಆಯ್ಕೆ, ಕ್ಯಾಶಿಂಗ್ ಮತ್ತು ಟೋಕನ್ ಬಜೆಟ್‌ಗಳ ಮೂಲಕ.

ಈ ಉತ್ಪಾದನಾ ಮಾದರಿಗಳು ನಿಮ್ಮ AI ಏಜೆಂಟ್‌ಗಳು ವ್ಯಾಪ್ತಿಯಲ್ಲಿ ವಿಶ್ವಾಸಾರ್ಹ, ಅಳೆಯಬಹುದಾದ ಮತ್ತು ವೆಚ್ಚದ ದೃಷ್ಟಿಯಿಂದ ಪರಿಣಾಮಕಾರಿಯಾಗಿರಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ಡಿಸ್ಕ್ಲೇಮರ್:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಗೆ ಪ್ರಯತ್ನಿಸಿದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸಮತ್ಯತೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ಮೂಲ ದಸ್ತಾವೇಜನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಅತ್ಯಂತ ಮುಖ್ಯವಾದ ಮಾಹಿತಿಗೂ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತಿದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಕೆಮಾಡುವುದರಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪಾಗಿ ಅರ್ಥಮಾಡಿಕೆಗಳು ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳಿಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
